# 05 — Reranking with CrossEncoders

**Repo:** ai-learning-notes | **Folder:** rag-and-retrieval  
**Built alongside:** local-rag-llm (github.com/RT91-data/local-rag-llm)

---

## What this notebook covers

- Bi-encoder vs CrossEncoder — the architectural difference and why it matters
- How CrossEncoder reads question + chunk jointly through attention
- Why CrossEncoder runs AFTER FAISS, not instead of it — the cost argument
- ms-marco-MiniLM-L-6-v2 — what the model name means, what it was trained on
- What CrossEncoder scores mean and how to interpret them
- Where CrossEncoder succeeds over pure vector search
- CrossEncoder failure modes
- How reranking improved local-rag-llm quality

**Pure Python only — no external dependencies.**

---
## 1. The Problem With Bi-Encoder Retrieval

In notebook 02 we saw that FAISS uses bi-encoder embeddings:  
- Question → encoder → vector_Q  
- Chunk → encoder → vector_C  
- Similarity = dot_product(vector_Q, vector_C)

**The fundamental limitation:**  
The question and chunk are encoded **independently**.  
They never interact with each other during encoding.  
The encoder cannot see both at the same time.

This means the model cannot reason about:  
- Whether a specific claim in the chunk answers the specific question  
- Whether the chunk's detail level matches what's being asked  
- Whether the chunk is topically adjacent but not actually an answer

**Classic failure:** A query about "how agents are identified" retrieves a chunk about  
"identity and access management" with high similarity — but the chunk is about access control,  
not about agent identity assignment. The vectors are close, but the chunk doesn't answer the question.

A CrossEncoder fixes this by reading both together.

In [ ]:
import math

# Demonstrating the bi-encoder failure case

query = "How are individual agents assigned unique identities?"

# These chunks are all semantically related to 'identity' — similar vectors
# But only ONE actually answers the question
candidate_chunks = [
    {
        "id": "A",
        "content": "Organisations must assign unique, cryptographic identities such as SPIFFE IDs to every individual agent.",
        "bi_encoder_sim": 0.88,   # high — 'unique identities' matches well
        "answers_question": True,
    },
    {
        "id": "B",
        "content": "Identity and Access Management covers authentication, authorisation, and audit across all system components.",
        "bi_encoder_sim": 0.86,   # high — 'identity' keyword dominates
        "answers_question": False,  # general IAM definition, not agent-specific
    },
    {
        "id": "C",
        "content": "The Pillar 5 IAM section covers the Confused Deputy problem and delegated vs agentic identity.",
        "bi_encoder_sim": 0.85,   # high — 'identity' again
        "answers_question": False,  # section header — no actual answer
    },
    {
        "id": "D",
        "content": "Access relies on Attribute-Based Access Control (ABAC) and JIT token downscoping.",
        "bi_encoder_sim": 0.79,   # lower — about access not identity assignment
        "answers_question": False,
    },
]

ranked_by_biencoder = sorted(candidate_chunks, key=lambda x: x["bi_encoder_sim"], reverse=True)

print(f"Query: '{query}'")
print()
print("BI-ENCODER RANKING (FAISS output)")
print(f"{'Rank':<5} {'Sim':>7} {'Answers?':>9}  Chunk")
print("-" * 75)
for rank, chunk in enumerate(ranked_by_biencoder, 1):
    answers = "✅ YES" if chunk["answers_question"] else "❌ No"
    print(f"{rank:<5} {chunk['bi_encoder_sim']:>7.3f} {answers:>9}  {chunk['content'][:55]}...")

print()
print("The correct chunk (A) is rank 1, but chunks B and C are close behind.")
print("If top_k=2, B and C both get into the context — adding noise.")
print("If top_k=1, we get lucky this time — but similarity scores are fragile.")
print("CrossEncoder can definitively separate A from B and C by reading both together.")

---
## 2. How a CrossEncoder Works

A CrossEncoder is a transformer that takes **both** the question and a candidate chunk  
as a single concatenated input:

```
[CLS] question [SEP] chunk [SEP]
     ↓
  Transformer encoder (all layers)
  — self-attention sees BOTH question and chunk tokens simultaneously
  — question tokens attend to chunk tokens and vice versa
     ↓
  [CLS] token final representation
     ↓
  Linear layer → single relevance score
```

**The key difference from bi-encoder:**  
In a bi-encoder, question and chunk tokens never interact through attention.  
In a CrossEncoder, every question token attends to every chunk token in every layer.  
The model can learn "this chunk uses 'SPIFFE IDs' which is the specific mechanism  
the question is asking about" — bi-encoder cannot make this connection.

**Why the CLS token?**  
The `[CLS]` token is a special token prepended to every input.  
After all transformer layers, its final hidden state aggregates information from  
the entire input sequence through attention. It's used as the sequence-level representation  
for classification tasks — including relevance scoring.

In [ ]:
# Illustrating what the CrossEncoder 'sees' vs what the bi-encoder sees

query   = "How are individual agents assigned unique identities?"
chunk_a = "Organisations must assign unique, cryptographic identities such as SPIFFE IDs to every individual agent."
chunk_b = "Identity and Access Management covers authentication, authorisation, and audit across all system components."

print("WHAT EACH ENCODER ARCHITECTURE SEES")
print("=" * 65)

print("\nBI-ENCODER (FAISS):")
print("  Step 1: Encode query INDEPENDENTLY")
print(f"    Input:  '{query[:60]}...'")
print(f"    Output: vector_Q = [0.12, -0.34, 0.87, ...] (768 dims)")
print()
print("  Step 2: Encode chunk A INDEPENDENTLY")
print(f"    Input:  '{chunk_a[:60]}...'")
print(f"    Output: vector_A = [0.15, -0.31, 0.84, ...] (768 dims)")
print()
print("  Step 3: Encode chunk B INDEPENDENTLY")
print(f"    Input:  '{chunk_b[:60]}...'")
print(f"    Output: vector_B = [0.11, -0.30, 0.83, ...] (768 dims)")
print()
print("  Comparison: dot(vector_Q, vector_A) vs dot(vector_Q, vector_B)")
print("  ⚠️  Query and chunk tokens NEVER interact — similarity is approximate")

print()
print("CROSSENCODER (Reranker):")
print("  Input A: [CLS] How are individual agents assigned unique identities? [SEP]")
print(f"           Organisations must assign unique cryptographic identities... [SEP]")
print("  → Self-attention: 'unique identities' in Q attends to 'unique cryptographic")
print("    identities' and 'SPIFFE IDs' in chunk A → high relevance score")
print()
print("  Input B: [CLS] How are individual agents assigned unique identities? [SEP]")
print(f"           Identity and Access Management covers authentication... [SEP]")
print("  → Self-attention: 'agents assigned' in Q finds no matching content in chunk B")
print("    → low relevance score")
print()
print("  ✅  Query and chunk tokens interact through all attention layers")
print("  ✅  Model learns: 'chunk A specifically answers the assignment question'")

---
## 3. Why CrossEncoder Runs AFTER FAISS — The Cost Argument

CrossEncoder is more accurate than bi-encoder similarity.  
So why not use it for retrieval directly and skip FAISS?

**The answer is computational cost.**  

FAISS similarity search is a single matrix multiplication.  
For 250 chunks: one forward pass computes similarity to all 250 at once — microseconds.

CrossEncoder requires a separate forward pass for **each** (query, chunk) pair.  
For 250 chunks: 250 forward passes — each taking ~80ms on CPU.

FAISS first narrows to 16 candidates.  
CrossEncoder then runs 16 forward passes — taking ~1.3 seconds.  
Versus 250 forward passes — taking ~20 seconds.

The two-stage architecture gives you accuracy close to full CrossEncoder search  
at a fraction of the cost — because FAISS is good enough to include the right chunks  
in its top-16, even if it doesn't rank them correctly.

In [ ]:
# Computational cost comparison

configs = [
    {
        "approach": "CrossEncoder over all chunks (naive)",
        "faiss_ms": 0,
        "ce_calls": 250,
        "ce_ms_each": 80,
        "accuracy": "100% (exact)",
    },
    {
        "approach": "FAISS only (no reranking)",
        "faiss_ms": 5,
        "ce_calls": 0,
        "ce_ms_each": 0,
        "accuracy": "~80% (approximate)",
    },
    {
        "approach": "FAISS k=16 → CrossEncoder top_k=5",
        "faiss_ms": 5,
        "ce_calls": 16,
        "ce_ms_each": 80,
        "accuracy": "~97% (FAISS rarely misses the true top-5)",
    },
    {
        "approach": "FAISS k=8 → CrossEncoder top_k=5 (old)",
        "faiss_ms": 5,
        "ce_calls": 8,
        "ce_ms_each": 80,
        "accuracy": "~93% (tighter candidate pool)",
    },
]

print(f"{'Approach':<42} {'Total ms':>9} {'CE calls':>9} {'Accuracy'}")
print("-" * 85)
for c in configs:
    total = c["faiss_ms"] + c["ce_calls"] * c["ce_ms_each"]
    current = " ← current" if "k=16" in c["approach"] else ""
    print(f"{c['approach']:<42} {total:>9}ms {c['ce_calls']:>9}  {c['accuracy']}{current}")

print()
print("Key insight: FAISS k=16 → CrossEncoder gives 97% of the accuracy")
print("of full CrossEncoder search at 6% of the cost (1285ms vs 20,000ms).")
print()
print("Why 97% and not 100%?")
print("FAISS might rank the true top-5 chunk at position #17 or #18 in rare cases.")
print("k=16 catches this for the vast majority of queries.")
print("k=32 would raise this to ~99% at double the CrossEncoder cost.")

---
## 4. ms-marco-MiniLM-L-6-v2 — What the Model Name Means

The CrossEncoder used in local-rag-llm is `cross-encoder/ms-marco-MiniLM-L-6-v2`.

Breaking down the name:

**ms-marco** — Microsoft MAchine Reading COmprehension dataset.  
A large-scale information retrieval benchmark with 8.8 million passages  
and 1 million real queries from Bing search logs.  
The model was trained to score (query, passage) relevance on this dataset.

**MiniLM** — A distilled version of a larger BERT model.  
MiniLM-L6 has 6 transformer layers (BERT-base has 12).  
Distillation transfers knowledge from a large teacher model to a small student model.  
Result: ~40% of the parameters, ~80% of the performance, ~3× the speed.

**v2** — Second version, improved training over v1.

**Why this model for local-rag-llm?**
- Runs on CPU in ~80ms per (query, chunk) pair — fast enough for 16 candidates
- Trained on real IR data — not fine-tuned on synthetic examples
- Open source, no API key, no cost per call
- ~22MB model size — negligible download and memory
- Strong performance on passage retrieval tasks

In [ ]:
# Understanding MiniLM distillation — why it's fast

models = [
    {
        "name": "BERT-large",
        "layers": 24,
        "hidden_size": 1024,
        "parameters_M": 340,
        "inference_ms": 450,
        "notes": "Full teacher model",
    },
    {
        "name": "BERT-base",
        "layers": 12,
        "hidden_size": 768,
        "parameters_M": 110,
        "inference_ms": 180,
        "notes": "Standard baseline",
    },
    {
        "name": "MiniLM-L12-v2",
        "layers": 12,
        "hidden_size": 384,
        "parameters_M": 33,
        "inference_ms": 120,
        "notes": "Smaller hidden — distilled",
    },
    {
        "name": "MiniLM-L6-v2",
        "layers": 6,
        "hidden_size": 384,
        "parameters_M": 22,
        "inference_ms": 80,
        "notes": "Used in local-rag-llm ← current",
    },
]

print(f"{'Model':<20} {'Layers':>7} {'Hidden':>7} {'Params M':>9} {'ms/call':>8}  Notes")
print("-" * 80)
for m in models:
    print(
        f"{m['name']:<20} {m['layers']:>7} {m['hidden_size']:>7} "
        f"{m['parameters_M']:>9} {m['inference_ms']:>8}  {m['notes']}"
    )

print()
bert_large_ms = 450
minilm_ms = 80
print(f"MiniLM-L6-v2 is {bert_large_ms/minilm_ms:.1f}× faster than BERT-large.")
print(f"For k=16 candidates: {16*minilm_ms}ms total reranking time.")
print(f"With BERT-large: {16*bert_large_ms}ms — too slow for interactive use.")
print()
print("Performance retention through distillation:")
print("  MiniLM-L6-v2 retains ~85-90% of BERT-base performance on MS MARCO")
print("  while being 5× smaller and 2× faster.")
print("  For document RAG (not web search), the gap is even smaller.")

---
## 5. What CrossEncoder Scores Mean

CrossEncoders trained on MS MARCO output raw logit scores — not probabilities.  
The scores are NOT bounded to 0-1. They can range roughly from -10 to +10.  
The absolute value of the score is not meaningful across different query-chunk pairs.  
What matters is the **relative ranking** within a single query's candidate set.

**Common mistake:** Treating CrossEncoder scores as relevance probabilities.  
A score of 5.2 does not mean "52% relevant".  
A score of 5.2 vs 3.1 means "chunk A is more relevant than chunk B for this query".

**In local-rag-llm:**  
Scores are stored in `doc.metadata["rerank_score"]` and displayed in the Sources expander.  
They are used only for sorting — the top `top_k=5` are kept regardless of absolute score value.

In [ ]:
# Simulating CrossEncoder score distribution
# Real scores from ms-marco-MiniLM-L-6-v2 look like this

import random
random.seed(7)

def mock_crossencoder_score(query, chunk):
    """
    NOT a real CrossEncoder. Simulates score range and distribution.
    Real model: sentence_transformers CrossEncoder.predict([(query, chunk)])
    """
    query_words = set(query.lower().split())
    chunk_words  = set(chunk.lower().split())
    overlap = len(query_words & chunk_words)
    # Simulate: more overlap + longer answer → higher score
    base = overlap * 1.2 + len(chunk) / 200
    noise = random.gauss(0, 0.5)
    return round(base + noise, 3)

query = "How are individual agents assigned unique identities?"

candidates = [
    "Organisations must assign unique cryptographic identities such as SPIFFE IDs to every individual agent in the system.",
    "Identity and Access Management covers authentication authorisation and audit across all system components.",
    "The Pillar 5 IAM section covers the Confused Deputy problem and delegated vs agentic identity.",
    "Access relies on Attribute-Based Access Control ABAC and JIT token downscoping for all agents.",
    "Zero Ambient Authority means agents must never inherit the developer's full administrative privileges.",
    "Table of contents: Introduction 6, Security 7, The Foundation 9, Sandboxes 13",  # junk
]

scores = [(chunk, mock_crossencoder_score(query, chunk)) for chunk in candidates]
scores.sort(key=lambda x: x[1], reverse=True)

print(f"Query: '{query}'")
print()
print("CROSSENCODER SCORES (simulated — real scores have similar distribution)")
print(f"{'Rank':<5} {'Score':>8}  Chunk")
print("-" * 75)
for rank, (chunk, score) in enumerate(scores, 1):
    kept = "✅" if rank <= 3 else "  "
    print(f"{rank:<5} {score:>8.3f} {kept} {chunk[:60]}...")

print()
print("Scores are NOT probabilities — they're logits.")
print("Only the RANKING matters, not absolute values.")
print("top_k=3 here would keep the top 3 regardless of score magnitude.")
print()
print("Note: junk TOC chunk scores low because it has no relevant content.")
print("CrossEncoder correctly demotes it — bi-encoder might not.")

---
## 6. What CrossEncoder Fixes That FAISS Cannot

Four specific failure modes where CrossEncoder consistently outperforms bi-encoder similarity:

In [ ]:
failures_fixed = [
    {
        "failure": "Topically adjacent but non-answering chunks",
        "example_query": "How are agents assigned identities?",
        "faiss_mistake": "Returns IAM section header (high sim) + ZAA chunk (high sim)",
        "ce_fix": "Correctly ranks SPIFFE ID assignment chunk #1, demotes section headers",
    },
    {
        "failure": "TOC and figure caption noise",
        "example_query": "What are the 7 pillars?",
        "faiss_mistake": "TOC chunk scores 0.88 — contains exact section title",
        "ce_fix": "TOC has no answer content — CrossEncoder assigns it a low score",
    },
    {
        "failure": "Partial answer chunks",
        "example_query": "What does the Green Team do?",
        "faiss_mistake": "Returns triad overview chunk (mentions Green Team once) + full description",
        "ce_fix": "Promotes the chunk with detailed Green Team description, demotes the mention",
    },
    {
        "failure": "Negation and contrast",
        "example_query": "What is NOT recommended for incident response?",
        "faiss_mistake": "Returns chunks about incident response (semantically close, ignores negation)",
        "ce_fix": "Reads 'killing the host container is NOT recommended' — correctly identifies it",
    },
]

print("WHAT CROSSENCODER FIXES OVER BI-ENCODER")
print("=" * 70)
for i, f in enumerate(failures_fixed, 1):
    print(f"\n{i}. {f['failure']}")
    print(f"   Query:       '{f['example_query']}'")
    print(f"   FAISS error: {f['faiss_mistake']}")
    print(f"   CE fix:      {f['ce_fix']}")

---
## 7. CrossEncoder Failure Modes

CrossEncoder is powerful but not perfect:

In [ ]:
failure_modes = [
    {
        "name": "Truncation on long chunks",
        "description": (
            "MiniLM-L6-v2 has a 512-token max input limit. "
            "Query + chunk must fit in 512 tokens. "
            "At chunk_size=2000 chars (~500 tokens) + query (~30 tokens), "
            "you're at the limit. Longer chunks get silently truncated."
        ),
        "impact": "Answer at the END of a long chunk may be cut off — invisible failure",
        "mitigation": "Keep chunk_size ≤ 1800 chars when using MiniLM, or use a 4096-token CrossEncoder",
    },
    {
        "name": "Domain mismatch",
        "description": (
            "Trained on MS MARCO (web search queries + Bing passages). "
            "Web search queries are short and informal. "
            "D365 ERP queries are technical and use domain-specific terminology."
        ),
        "impact": "May underrank chunks containing correct ERP technical answers",
        "mitigation": "Fine-tune on domain-specific (query, relevant_chunk, irrelevant_chunk) triples",
    },
    {
        "name": "Computationally expensive at scale",
        "description": "80ms per call on CPU. Fine for k=16. Problematic if k grows.",
        "impact": "If FAISS k increases to 32+, reranking latency becomes noticeable",
        "mitigation": "Run on GPU (5-10× faster) or use a smaller model (TinyBERT)",
    },
    {
        "name": "Cannot fix missing candidates",
        "description": (
            "CrossEncoder only re-ranks what FAISS + BM25 retrieved. "
            "If the correct chunk is not in the top-16 from FAISS, "
            "CrossEncoder cannot surface it."
        ),
        "impact": "FAISS k must be large enough to include the true answer chunk",
        "mitigation": "Tune FAISS k using RAGAS context_recall — if low, increase k",
    },
]

print("CROSSENCODER FAILURE MODES")
print("=" * 70)
for i, f in enumerate(failure_modes, 1):
    print(f"\n{i}. {f['name']}")
    print(f"   What:    {f['description']}")
    print(f"   Impact:  {f['impact']}")
    print(f"   Fix:     {f['mitigation']}")

---
## 8. Reranking in the local-rag-llm Code

The CrossEncoder is integrated in `app.py` as follows:

```python
from sentence_transformers import CrossEncoder

@st.cache_resource
def load_reranker():
    # Loads model once, shared across all users (22MB)
    return CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

def rerank_chunks(query: str, docs: list, top_k: int = 5) -> list:
    if not docs or len(docs) <= 1:
        return docs
    
    # Build (query, chunk) pairs for CrossEncoder
    pairs = [[query, doc.page_content] for doc in docs]
    
    # One forward pass per pair
    scores = reranker.predict(pairs)
    
    # Sort by score descending
    scored_docs = sorted(zip(scores, docs), key=lambda x: x[0], reverse=True)
    
    # Store score in metadata for display
    for score, doc in scored_docs:
        doc.metadata["rerank_score"] = round(float(score), 3)
    
    # Return top_k
    return [doc for _, doc in scored_docs[:top_k]]
```

**Two things to note:**
1. `@st.cache_resource` ensures the 22MB model loads once — not on every query
2. The score is stored as `rerank_score` and shown in the Sources expander — this is useful for debugging which chunks were highly ranked and why

In [ ]:
# Full reranking pipeline simulation — from merged candidates to final top_k

random.seed(42)

query = "What problem does the Vibe Diff solve?"

merged_candidates = [
    "The Vibe Diff shows the developer exactly how their fuzzy intent maps to proposed execution steps.",
    "Simple approval gates cause confirmation fatigue — developers blindly authorise code they don't understand.",
    "The It Works Ship It fallacy: developers trust generated code because it compiles without errors.",
    "High-stakes actions such as modifying production databases require explicit verification.",
    "Table of contents: Elicitation MFA Challenges and the Vibe Diff 19",  # junk
    "Cryptographic Hardware MFA requires physical touch of a USB security key.",
    "Before a critical tool runs an Evaluator Quorum intercepts and translates the code to plain English.",
    "Zero Ambient Authority means agents must never inherit full administrative privileges.",  # off-topic
]

# Simulate CrossEncoder scoring
def sim_ce_score(query, chunk):
    qw = set(query.lower().split())
    cw = set(chunk.lower().split())
    overlap = len(qw & cw)
    length_bonus = min(len(chunk) / 150, 1.5)
    noise = random.gauss(0, 0.3)
    # Penalise junk chunks
    junk_penalty = -3.0 if "table of contents" in chunk.lower() else 0
    return round(overlap * 0.8 + length_bonus + noise + junk_penalty, 3)

scores = [(chunk, sim_ce_score(query, chunk)) for chunk in merged_candidates]
scores_sorted = sorted(scores, key=lambda x: x[1], reverse=True)

TOP_K = 4

print(f"Query: '{query}'")
print(f"Candidates in: {len(merged_candidates)} | top_k: {TOP_K}")
print()
print(f"{'Rank':<5} {'Score':>8} {'Kept':>6}  Chunk")
print("-" * 80)
for rank, (chunk, score) in enumerate(scores_sorted, 1):
    kept = "✅ YES" if rank <= TOP_K else "   No"
    print(f"{rank:<5} {score:>8.3f} {kept}  {chunk[:60]}...")

print()
print(f"Top {TOP_K} chunks sent to Claude:")
for rank, (chunk, score) in enumerate(scores_sorted[:TOP_K], 1):
    print(f"  {rank}. [{score:.3f}] {chunk[:60]}...")

---
## 9. Summary

| Concept | Key point |
|---|---|
| **Bi-encoder** | Question and chunk encoded independently — fast but approximate |
| **CrossEncoder** | Question and chunk encoded jointly — slow but accurate |
| **Joint attention** | Every Q token attends to every chunk token — finds specific answer match |
| **Two-stage pipeline** | FAISS (fast, approximate) → CrossEncoder (slow, precise) |
| **k=16 → top_k=5** | FAISS retrieves 16, CrossEncoder keeps best 5 |
| **ms-marco-MiniLM-L-6-v2** | 22MB, 6-layer distilled BERT, trained on Bing search data |
| **Scores are logits** | Not probabilities — only relative ranking matters |
| **Fixes TOC noise** | TOC chunks have no answer content → CrossEncoder scores them low |
| **Cannot fix missing** | If FAISS misses the chunk entirely, CrossEncoder cannot surface it |
| **512 token limit** | chunk_size=2000 chars (~500 tokens) approaches the limit — watch for truncation |

**The reranking insight:**  
FAISS and BM25 retrieve. CrossEncoder selects.  
FAISS is fast and broad — cast a wide net.  
CrossEncoder is slow and precise — pick the best from the net.  
Neither can do the other's job efficiently.

---
## Next Notebooks

- **06** — RAG evaluation with RAGAS (metrics, golden dataset, reading and acting on results)
- **07** — RAG security (injection attacks, 3-layer defence, fail-open vs fail-closed)
- **08** — Advanced RAG patterns (parent-document, multi-query, self-querying)